# Vesper Capital — Hourly Balance Fetch: Accepted Wallets

Fetches the full hourly balance history (Jan 2023 - Dec 2024) for wallets accepted by the wallet search pipeline, then merges with the existing top-200 balance store to create an expanded wallet pool.

**Workflow:**
1. Load accepted wallets from `accepted_wallets.csv`
2. Identify which are not already in the top-200 balance store
3. Fetch hourly balances for new wallets (restart-safe)
4. Merge with existing store → `balance_store_hourly_combined.csv`
5. Retrain walk-forward model with expanded pool

**Quality filter:** set `FETCH_QUALITY = "high_med"` to fetch only wallets with ≥10 detected trades (recommended), or `"all"` for every accepted wallet.

**Restart-safe:** tracks fetched (address, hour) pairs. Re-run fetch cell to continue from where it stopped.

## 1. Configuration & cost estimate

In [1]:
from claude_refactor_core_utils import *
import pandas as pd
import numpy as np
import os
import hashlib
from datetime import date, timedelta
from pathlib import Path

ALCHEMY_API_KEY   = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR          = Path('/home/colin/projects/blockchain_alpha')
SEARCH_DIR        = BASE_DIR / 'data/wallet_search'
ACCEPTED_CSV      = SEARCH_DIR / 'accepted_wallets.csv'
HOURLY_BAL_CSV    = BASE_DIR / 'data/balance_store_hourly_top200.csv'
ACCEPTED_BAL_CSV  = BASE_DIR / 'data/balance_store_accepted_wallets.csv'
BLOCK_MAP_CSV     = BASE_DIR / 'data/hourly_block_map.csv'

# ── Fetch window — match existing balance store ────────────────
FETCH_START = date(2023, 1, 1)
FETCH_END   = date(2024, 12, 31)

# ── Fetch params ───────────────────────────────────────────────
BATCH_SIZE      = 100
MAX_CONCURRENCY = 30
RPS             = 25.0

# ── Cost estimate ──────────────────────────────────────────────
cu_per_balance      = 19
cu_rate_per_million = 0.45

n_hours = (pd.Timestamp(FETCH_END) - pd.Timestamp(FETCH_START)).days * 24

# Load accepted wallets
accepted_df = pd.read_csv(ACCEPTED_CSV)
accepted_df['address'] = accepted_df['address'].str.lower().str.strip()

# Focus on HIGH and MEDIUM quality
high_med = accepted_df[accepted_df['n_trades'] >= 10].copy()
all_accepted = accepted_df.copy()

print("=" * 60)
print("HOURLY BALANCE FETCH — ACCEPTED WALLETS")
print("=" * 60)
print()
print(f"Fetch window:       {FETCH_START} to {FETCH_END}")
print(f"Hours to fetch:     {n_hours:,}")
print()
print(f"Accepted wallets total:        {len(accepted_df)}")
print(f"HIGH quality (>=20t, >0.5%e):  {len(accepted_df[(accepted_df['n_trades']>=20) & (accepted_df['weighted_edge']>0.5)])}")
print(f"MED  quality (10-19 trades):   {len(accepted_df[(accepted_df['n_trades']>=10) & (accepted_df['n_trades']<20)])}")
print(f"LOW  quality (<10 trades):     {len(accepted_df[accepted_df['n_trades']<10])}")
print()

# Cost estimates
for label, wallet_list in [
    ("HIGH+MED only (>=10 trades)", high_med),
    ("ALL accepted",                all_accepted),
]:
    n_w    = len(wallet_list)
    est_cu = n_w * n_hours * cu_per_balance
    print(f"{label}:")
    print(f"  Wallets:   {n_w}")
    print(f"  Pairs:     {n_w * n_hours:,}")
    print(f"  Est. CUs:  {est_cu:,}  (theoretical)")
    print(f"  Est. cost: ${est_cu/1e6*cu_rate_per_million:.2f}  (theoretical)")
    print(f"  Empirical cost (~320 CU/wallet/month): "
          f"${n_w * 12 * 320 / 1e6 * cu_rate_per_million:.2f}")
    print()

print("Recommendation: fetch HIGH+MED first, then LOW if budget allows.")
print("Set FETCH_QUALITY below before running.")
print()
print("  FETCH_QUALITY = 'high_med'  → fetch wallets with n_trades >= 10")
print("  FETCH_QUALITY = 'all'       → fetch all accepted wallets")
FETCH_QUALITY = 'high_med'   # change to 'all' for all accepted wallets
print()
print(f"Current setting: FETCH_QUALITY = '{FETCH_QUALITY}'")


HOURLY BALANCE FETCH — ACCEPTED WALLETS

Fetch window:       2023-01-01 to 2024-12-31
Hours to fetch:     17,520

Accepted wallets total:        113
HIGH quality (>=20t, >0.5%e):  52
MED  quality (10-19 trades):   16
LOW  quality (<10 trades):     0

HIGH+MED only (>=10 trades):
  Wallets:   113
  Pairs:     1,979,760
  Est. CUs:  37,615,440  (theoretical)
  Est. cost: $16.93  (theoretical)
  Empirical cost (~320 CU/wallet/month): $0.20

ALL accepted:
  Wallets:   113
  Pairs:     1,979,760
  Est. CUs:  37,615,440  (theoretical)
  Est. cost: $16.93  (theoretical)
  Empirical cost (~320 CU/wallet/month): $0.20

Recommendation: fetch HIGH+MED first, then LOW if budget allows.
Set FETCH_QUALITY below before running.

  FETCH_QUALITY = 'high_med'  → fetch wallets with n_trades >= 10
  FETCH_QUALITY = 'all'       → fetch all accepted wallets

Current setting: FETCH_QUALITY = 'high_med'


## 2. Load block map

In [2]:
# Load block map for the fetch window
print("Loading block map...")
bm_df = pd.read_csv(BLOCK_MAP_CSV)
bm_df['timestamp'] = pd.to_datetime(bm_df['timestamp']).dt.floor('h')
bm_df['block_number'] = bm_df['block_number'].astype(int)

# Filter to fetch window
bm_window = bm_df[
    (bm_df['timestamp'] >= pd.Timestamp(FETCH_START)) &
    (bm_df['timestamp'] <= pd.Timestamp(FETCH_END))
].reset_index(drop=True)

print(f"Block map entries in window: {len(bm_window):,}")
print(f"Range: {bm_window['timestamp'].min()} to {bm_window['timestamp'].max()}")

# Sanity check
diffs = bm_window['block_number'].diff().dropna()
print(f"Mean blocks/hour: {diffs.mean():.0f}  (expected ~300)")
print(f"Monotonic:        {(diffs > 0).all()}")


Loading block map...
Block map entries in window: 17,521
Range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Mean blocks/hour: 297  (expected ~300)
Monotonic:        True


## 3. Build wallet fetch list

Identifies which accepted wallets are not already in the existing top-200 balance store — no duplicate fetching.

In [3]:
# Build wallet list based on FETCH_QUALITY setting
accepted_df = pd.read_csv(ACCEPTED_CSV)
accepted_df['address'] = accepted_df['address'].str.lower().str.strip()

if FETCH_QUALITY == 'high_med':
    wallet_df   = accepted_df[accepted_df['n_trades'] >= 10].copy()
    fetch_label = "HIGH+MED quality"
else:
    wallet_df   = accepted_df.copy()
    fetch_label = "ALL accepted"

wallet_list = wallet_df['address'].tolist()

# Check which wallets are already in the existing balance store
already_fetched_wallets = set()
if ACCEPTED_BAL_CSV.exists():
    existing = pd.read_csv(ACCEPTED_BAL_CSV, usecols=['address'])
    already_fetched_wallets = set(existing['address'].str.lower().str.strip().unique())
    print(f"Already fetched: {len(already_fetched_wallets)} wallets in {ACCEPTED_BAL_CSV.name}")

# Also check existing hourly balance store
if HOURLY_BAL_CSV.exists():
    existing_top200 = pd.read_csv(HOURLY_BAL_CSV, usecols=['address'])
    top200_wallets  = set(existing_top200['address'].str.lower().str.strip().unique())
    in_top200       = [w for w in wallet_list if w in top200_wallets]
    not_in_top200   = [w for w in wallet_list if w not in top200_wallets]
    print(f"Already in top200 balance store: {len(in_top200)}")
    print(f"New wallets to fetch:            {len(not_in_top200)}")
    wallet_list = not_in_top200  # only fetch wallets not already in the store
else:
    print("No existing balance store found — fetching all")

print()
print(f"Fetch plan: {fetch_label}")
print(f"  Wallets to fetch:  {len(wallet_list)}")
print(f"  Hours per wallet:  {n_hours:,}")
print(f"  Total pairs:       {len(wallet_list) * n_hours:,}")
print()
if len(wallet_list) == 0:
    print("Nothing to fetch — all wallets already in balance store.")
else:
    print("Wallets to fetch:")
    for addr in wallet_list[:10]:
        row = wallet_df[wallet_df['address']==addr].iloc[0] if addr in wallet_df['address'].values else None
        edge = f"{row['weighted_edge']:+.3f}%" if row is not None else ""
        n    = f"n={int(row['n_trades'])}" if row is not None else ""
        print(f"  {addr}  {edge}  {n}")
    if len(wallet_list) > 10:
        print(f"  ... and {len(wallet_list)-10} more")


Already in top200 balance store: 0
New wallets to fetch:            113

Fetch plan: HIGH+MED quality
  Wallets to fetch:  113
  Hours per wallet:  17,520
  Total pairs:       1,979,760

Wallets to fetch:
  0x90843bb8e525ee77aea03b97a8826b6ee1bab43e  +5.308%  n=11
  0x50553500f157e5c3b0bf4ddfd64dd8229492883d  +5.112%  n=83
  0x86a067030a9668c13ff2a8c4d5415afc776d4c63  +3.461%  n=15
  0xa26e73c8e9507d50bf808b7a2ca9d5de4fcc4a04  +3.195%  n=28
  0x87d18ee84e8f4f5709cbf3500179a4c601da12ce  +2.247%  n=22
  0xd0e83490a55e671b3cf39cd027f0f9035ec5a1ef  +2.126%  n=49
  0x11ec2540d694a7e3db427ae3ca989b01d3d24dd7  +2.107%  n=72
  0xb01007916cf90c1610e7429eaee53b2c0a4588a8  +1.942%  n=75
  0xcae89544ee4d7bd04d60c9c312cfbeebbbf28f6c  +1.875%  n=45
  0xa760c65ee687c854c9f20d90d9e5c7d27eb742b8  +1.815%  n=77
  ... and 103 more


## 4. Fetch hourly balances

**Restart-safe** — re-run this cell to continue from where it stopped.

Progress printed every 24 hours (1 day of data).

In [4]:
# ================================================================
# Hourly balance fetch for accepted wallets
# Restart-safe — tracks already-fetched (address, hour) pairs
# Appends to ACCEPTED_BAL_CSV
# ================================================================

if len(wallet_list) == 0:
    print("Nothing to fetch.")
else:
    # Load already-fetched pairs for warm start
    already_fetched_pairs = set()
    if ACCEPTED_BAL_CSV.exists():
        existing_bal = pd.read_csv(ACCEPTED_BAL_CSV, usecols=['address','date'])
        existing_bal['date']    = pd.to_datetime(existing_bal['date'], format='mixed').dt.floor('h')
        existing_bal['address'] = existing_bal['address'].str.lower().str.strip()
        already_fetched_pairs   = set(zip(existing_bal['address'], existing_bal['date']))
        print(f"Warm start: {len(already_fetched_pairs):,} pairs already fetched")

    total_pairs   = len(wallet_list) * n_hours
    already_count = sum(
        1 for addr in wallet_list
        for h in pd.date_range(start=FETCH_START, end=FETCH_END, freq='h')
        if (addr, pd.Timestamp(h).floor('h')) in already_fetched_pairs
    )
    print(f"Pairs to fetch: {total_pairs - already_count:,} of {total_pairs:,}")
    print()

    # Build reverse block lookup
    block_to_hour = dict(zip(bm_window['block_number'], bm_window['timestamp']))

    fetch_count = 0
    for hour_idx, hour in enumerate(
        pd.date_range(start=FETCH_START, end=FETCH_END, freq='h')
    ):
        hour_norm = pd.Timestamp(hour).floor('h')

        # Find block for this hour
        block_row = bm_window[bm_window['timestamp'] == hour_norm]
        if len(block_row) == 0:
            continue
        block_num = int(block_row['block_number'].iloc[0])

        # Build schedule for wallets not yet fetched at this hour
        sched_rows = [
            {"address": addr, "date": hour_norm}
            for addr in wallet_list
            if (addr, hour_norm) not in already_fetched_pairs
        ]

        if not sched_rows:
            continue

        sched_df   = pd.DataFrame(sched_rows)
        hour_bm_df = pd.DataFrame([{
            "timestamp": hour_norm, "block_number": block_num
        }])

        try:
            bal = fetch_balances_for_schedule(
                schedule        = sched_df,
                block_df        = hour_bm_df,
                alchemy_api_key = ALCHEMY_API_KEY,
                batch_size      = BATCH_SIZE,
                max_concurrency = MAX_CONCURRENCY,
                rps             = RPS,
                progress_every  = 99999,
                block_date_col  = "timestamp",
                label           = f"accepted-{str(hour_norm)[:13]}",
            )
        except Exception as e:
            print(f"  Error at {hour_norm}: {e}")
            continue

        if bal is not None and not bal.empty:
            # Fix date to use hour timestamp
            bal['date'] = hour_norm
            bal.to_csv(
                ACCEPTED_BAL_CSV,
                mode='a',
                header=not ACCEPTED_BAL_CSV.exists(),
                index=False
            )
            # Update warm start set
            for addr in bal['address'].str.lower().str.strip():
                already_fetched_pairs.add((addr, hour_norm))
            fetch_count += len(bal)

        # Progress every 24 hours (1 day)
        if (hour_idx + 1) % 24 == 0:
            pct = (hour_idx + 1) / n_hours * 100
            print(f"  [{pct:5.1f}%]  {str(hour_norm)[:13]}  "
                  f"rows written: {fetch_count:,}")

    print(f"\nFetch complete. {fetch_count:,} new rows written to {ACCEPTED_BAL_CSV.name}")


Pairs to fetch: 1,979,760 of 1,979,760

[accepted-2023-01-01 00] 2/2 batches (100.0%)
[accepted-2023-01-01 01] 2/2 batches (100.0%)
[accepted-2023-01-01 02] 2/2 batches (100.0%)
[accepted-2023-01-01 03] 2/2 batches (100.0%)
[accepted-2023-01-01 04] 2/2 batches (100.0%)
[accepted-2023-01-01 05] 2/2 batches (100.0%)
[accepted-2023-01-01 06] 2/2 batches (100.0%)
[accepted-2023-01-01 07] 2/2 batches (100.0%)
[accepted-2023-01-01 08] 2/2 batches (100.0%)
[accepted-2023-01-01 09] 2/2 batches (100.0%)
[accepted-2023-01-01 10] 2/2 batches (100.0%)
[accepted-2023-01-01 11] 2/2 batches (100.0%)
[accepted-2023-01-01 12] 2/2 batches (100.0%)
[accepted-2023-01-01 13] 2/2 batches (100.0%)
[accepted-2023-01-01 14] 2/2 batches (100.0%)
[accepted-2023-01-01 15] 2/2 batches (100.0%)
[accepted-2023-01-01 16] 2/2 batches (100.0%)
[accepted-2023-01-01 17] 2/2 batches (100.0%)
[accepted-2023-01-01 18] 2/2 batches (100.0%)
[accepted-2023-01-01 19] 2/2 batches (100.0%)
[accepted-2023-01-01 20] 2/2 batches (10

## 5. Verify & merge with existing balance store

Checks coverage and merges accepted wallet balances with the existing top-200 store into `balance_store_hourly_combined.csv`.

In [5]:
# ================================================================
# Verify fetch and merge with existing balance store
# ================================================================

print("Verifying accepted wallet balance store...")
if not ACCEPTED_BAL_CSV.exists():
    print("File not found — run fetch cell first.")
else:
    bal_df = pd.read_csv(ACCEPTED_BAL_CSV)
    bal_df['date']    = pd.to_datetime(bal_df['date'], format='mixed').dt.floor('h')
    bal_df['address'] = bal_df['address'].str.lower().str.strip()

    # Deduplicate
    bal_df = (bal_df
              .sort_values(['address','date','block_number'])
              .groupby(['address','date'], as_index=False)
              .tail(1)
              .reset_index(drop=True))

    expected_pairs = len(wallet_list) * n_hours
    coverage       = len(bal_df) / expected_pairs * 100 if expected_pairs > 0 else 0

    print(f"Rows:             {len(bal_df):,}")
    print(f"Unique addresses: {bal_df['address'].nunique()}")
    print(f"Date range:       {bal_df['date'].min()} to {bal_df['date'].max()}")
    print(f"Coverage:         {coverage:.1f}%")
    print()

    # Rows per wallet
    per_wallet = bal_df.groupby('address').size().sort_values(ascending=False)
    print("Rows per wallet (sample):")
    print(per_wallet.head(10).to_string())
    print()

    # ── Merge with existing top200 balance store ───────────────
    print("Merging with existing top200 balance store...")
    if HOURLY_BAL_CSV.exists():
        top200_df = pd.read_csv(HOURLY_BAL_CSV)
        top200_df['date']    = pd.to_datetime(top200_df['date'], format='mixed').dt.floor('h')
        top200_df['address'] = top200_df['address'].str.lower().str.strip()

        # Combine
        combined = pd.concat([top200_df, bal_df], ignore_index=True)
        combined = (combined
                    .sort_values(['address','date','block_number'])
                    .groupby(['address','date'], as_index=False)
                    .tail(1)
                    .reset_index(drop=True))

        COMBINED_BAL_CSV = BASE_DIR / 'data/balance_store_hourly_combined.csv'
        combined.to_csv(COMBINED_BAL_CSV, index=False)

        print(f"Combined balance store:")
        print(f"  Wallets:    {combined['address'].nunique()}")
        print(f"  Rows:       {len(combined):,}")
        print(f"  Saved to:   {COMBINED_BAL_CSV.name}")
        print()
        print("Next steps:")
        print("  1. Update HOURLY_BALANCE_CSV in walkforward_validation.ipynb")
        print("     to point to balance_store_hourly_combined.csv")
        print("  2. Rerun walk-forward model with expanded wallet pool")
        print("  3. Compare results vs current 200-wallet baseline")
    else:
        print("No existing top200 balance store found.")
        print(f"Use {ACCEPTED_BAL_CSV.name} directly as the balance store.")


Verifying accepted wallet balance store...
Rows:             1,979,873
Unique addresses: 113
Date range:       2023-01-01 00:00:00 to 2024-12-31 00:00:00
Coverage:         100.0%

Rows per wallet (sample):
address
0x016606acc6b0cfe537acc221e3bf1bb44b4049ee    17521
0x05f0af42ed9f88d3d68d2f2a9d4a7d28af3c5e2b    17521
0x082d2397c5e3c5300d31f52bd056169882b8ba1c    17521
0x08d36479c6aa638c404925d34d42804b34fd7229    17521
0x09f96d8da0ac189a1ead263e1ebeb25c76fcb0bc    17521
0x0ab3fbc9025ece0ea4e0f9d29fbaa94b70923e37    17521
0x0c40bb77dc1144fd99eed98e1d6d76203508bf2c    17521
0x11ec2540d694a7e3db427ae3ca989b01d3d24dd7    17521
0x14222b470f809f1cfe7a6a01646b6e79cecc584f    17521
0x147ac0b39675769e55a0f0e7fdd3641b47963661    17521

Merging with existing top200 balance store...
Combined balance store:
  Wallets:    314
  Rows:       5,501,594
  Saved to:   balance_store_hourly_combined.csv

Next steps:
  1. Update HOURLY_BALANCE_CSV in walkforward_validation.ipynb
     to point to balance_stor

## 6. Next steps

In [6]:
# ================================================================
# Next steps summary
# ================================================================

print("=" * 60)
print("FETCH COMPLETE — NEXT STEPS")
print("=" * 60)
print()
print("1. VERIFY the combined balance store:")
print("   balance_store_hourly_combined.csv")
print(f"   Expected: ~{(200 + len(wallet_list)):,} wallets × {n_hours:,} hours")
print()
print("2. UPDATE walkforward_validation.ipynb config:")
print("   Change HOURLY_BALANCE_CSV to:")
print("   '/home/colin/projects/blockchain_alpha/data/balance_store_hourly_combined.csv'")
print()
print("3. RERUN walk-forward model")
print("   Run cells 1-6 in walkforward_validation.ipynb")
print("   Key metrics to compare:")
print("   - Non-November mean edge (was +1.005%/hr)")
print("   - T-test p-value (was p=0.028)")
print("   - Positive non-November folds (was 9/12)")
print()
print("4. IF improved: update combined_model.ipynb to use new balance store")
print("   Rerun capital simulation")
print("   Expected improvement: more active months, higher edge, better Sharpe")
print()
print("5. CONTINUE wallet search:")
print("   Run wallet_search_v3.ipynb on remaining candidates")
print("   Add newly accepted wallets to combined balance store")
print("   Retrain periodically as pool grows")
print()

if ACCEPTED_BAL_CSV.exists():
    bal_check = pd.read_csv(ACCEPTED_BAL_CSV)
    print(f"Accepted wallet balance store: {len(bal_check):,} rows")
    print(f"File: {ACCEPTED_BAL_CSV}")
if Path(BASE_DIR / 'data/balance_store_hourly_combined.csv').exists():
    combined_check = pd.read_csv(
        BASE_DIR / 'data/balance_store_hourly_combined.csv',
        usecols=['address']
    )
    print(f"Combined balance store: {combined_check['address'].nunique()} wallets")
    print(f"File: balance_store_hourly_combined.csv")


FETCH COMPLETE — NEXT STEPS

1. VERIFY the combined balance store:
   balance_store_hourly_combined.csv
   Expected: ~313 wallets × 17,520 hours

2. UPDATE walkforward_validation.ipynb config:
   Change HOURLY_BALANCE_CSV to:
   '/home/colin/projects/blockchain_alpha/data/balance_store_hourly_combined.csv'

3. RERUN walk-forward model
   Run cells 1-6 in walkforward_validation.ipynb
   Key metrics to compare:
   - Non-November mean edge (was +1.005%/hr)
   - T-test p-value (was p=0.028)
   - Positive non-November folds (was 9/12)

4. IF improved: update combined_model.ipynb to use new balance store
   Rerun capital simulation
   Expected improvement: more active months, higher edge, better Sharpe

5. CONTINUE wallet search:
   Run wallet_search_v3.ipynb on remaining candidates
   Add newly accepted wallets to combined balance store
   Retrain periodically as pool grows

Accepted wallet balance store: 1,979,873 rows
File: /home/colin/projects/blockchain_alpha/data/balance_store_accepted